In [39]:
from pathlib import Path
import os

print("Working directory:", os.getcwd())

for folder in [
    "generated_samples",
    "confusable_negatives",
    "negative_datasets/speech/training",
    "negative_datasets/dinner_party/training",
    "negative_datasets/no_speech/training",
    "negative_datasets/dinner_party_eval/testing_ambient",
]:
    path = Path("/content") / folder
    print(f"{folder}: {'OK' if path.is_dir() else 'MISSING'}")

Working directory: /content
generated_samples: OK
confusable_negatives: OK
negative_datasets/speech/training: OK
negative_datasets/dinner_party/training: OK
negative_datasets/no_speech/training: OK
negative_datasets/dinner_party_eval/testing_ambient: OK


# microWakeWord — Train any wake word

A single-notebook trainer for custom wake words on ESPHome `micro_wake_word` devices
(M5Stack Atom Echo, Voice PE, etc).

Two modes:
- **`generate`** — Piper TTS generates ~30k positive samples + your confusables in Colab.
  Easiest path. Requires an IPA pronunciation of your wake word.
- **`bundle`** — You upload a zip with your own samples (real recordings, ElevenLabs
  voices, accent-matched TTS). Higher quality but you do the prep work.

## Required runtime
**Runtime → Change runtime type → A100 GPU + High-RAM**.
T4 OOMs during validation. A100 + High-RAM gives 40 GB VRAM + 85 GB system RAM.

## What you get
A `<output_name>.tflite` (~60 KB) + companion `.json` manifest, ready to drop into
your ESPHome config under `micro_wake_word: models:`.

## Workflow
1. Edit the **CONFIGURE HERE** cell below for your wake word
2. **Runtime → Run all**, walk away ~45 minutes
3. Find `<output_name>.tflite` + `.json` in your Drive folder when it's done
4. Test on hardware — likely needs manifest tuning (cutoff, sliding_window) for
   your specific model. See the deployment notes at the end.

Built from working production deployment of "Hey Harold" — all known
upstream bugs are patched in this notebook.


In [33]:
# ╔═══════════════════════════════════════════════════════════════════════╗
# ║                    CONFIGURE YOUR WAKE WORD HERE                       ║
# ╚═══════════════════════════════════════════════════════════════════════╝

# ─── Wake word identity ───
WAKE_WORD = "Artemis"           # Human-readable name (shown in HA)
OUTPUT_NAME = "artemis"         # Filename (no spaces, lowercase)
AUTHOR = "lizaoreo"               # Goes in the manifest
AUTHOR_WEBSITE = "https://github.com/lizaoreo"

# ─── Mode ───
MODE = "generate"   # "generate" (Piper makes samples) | "bundle" (you uploaded a zip)

# ─── Drive folder (created if missing) ───
DRIVE_FOLDER = "wakeword_training_artemis"

# ─── If MODE == "generate" ───
# Look up IPA for your wake word: https://www.internationalphoneticassociation.org/IPA-chart
# or use eSpeak: `espeak-ng -q --ipa "Hey Harold"` on Linux
WAKE_WORD_IPA_US = "ˈɑɹtəmɪs"   # US English pronunciation
WAKE_WORD_IPA_UK = "ˈɑɹtɪmɪs"   # UK English (set to None to skip second pass)
SAMPLES_US = 30000                  # ~12 min on T4, ~7 min on A100
SAMPLES_UK = 6000                   # 0 to skip UK pass

# Confusable phrases — should NOT trigger your wake word.
# Include: phonetic neighbors, the bare name without prefix, common
# false-trigger phrases like other assistant names.
CONFUSABLE_PHRASES = [
    # General "hey X" near-misses
    "hey there", "hey you", "hey y'all", "hey now",
    # Similar names and words
    "Arthur",
    "Artie",
    "Artist",
    "Artists",
    "Artistic",
    "Art is",
    "Artem",
    # Similar conversational phrases
    "Our team",
    "Our teams",
    "Our time",
    "Are they missing",
    "Part of this",
    "Pardon me",
    "Hard to miss",
    "Hard times",
]
SAMPLES_PER_CONFUSABLE = 1000   # 500 = light, 1000 = recommended, 2000 = max

# ─── If MODE == "bundle" ───
# Expected: <DRIVE_FOLDER>/data_bundle.zip with this layout:
#   generated_samples/        positive samples (.wav, 16 kHz mono)
#   real_recordings/          (optional) real mic recordings
#   confusable_negatives/     (optional) hard negative samples
BUNDLE_NAME = "data_bundle.zip"

# ─── Manifest tuning (works well as defaults; tune after on-device testing) ───
PROBABILITY_CUTOFF = 0.85          # 0.5 = too lenient, 0.97 = okay_nabu strict
SLIDING_WINDOW_SIZE = 5            # 5 = okay_nabu default; higher = slower fire
TENSOR_ARENA_SIZE = 50000          # 30000 works, 50000 has more headroom

# ─── Languages this model is trained for (manifest metadata) ───
TRAINED_LANGUAGES = ["en"]         # add "it", "es" etc if you have multi-accent samples

# ╔═══════════════════════════════════════════════════════════════════════╗
# ║                           END OF CONFIG                                ║
# ╚═══════════════════════════════════════════════════════════════════════╝
print(f"Training '{WAKE_WORD}' as {OUTPUT_NAME} in mode={MODE!r}")
print(f"Output → /content/drive/MyDrive/{DRIVE_FOLDER}/{OUTPUT_NAME}.tflite")


Training 'Artemis' as artemis in mode='generate'
Output → /content/drive/MyDrive/wakeword_training_artemis/artemis.tflite


In [6]:
# === Mount Drive ===
from google.colab import drive
import os
drive.mount('/content/drive')

DRIVE_DIR = f'/content/drive/MyDrive/{DRIVE_FOLDER}'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive folder: {DRIVE_DIR}')
if MODE == 'bundle':
    BUNDLE_PATH = f'{DRIVE_DIR}/{BUNDLE_NAME}'
    assert os.path.exists(BUNDLE_PATH), (
        f'MODE=bundle but {BUNDLE_PATH} does not exist. Upload your data zip there.')
    print(f'Found bundle: {os.path.getsize(BUNDLE_PATH)/1024/1024:.1f} MB')


Mounted at /content/drive
Drive folder: /content/drive/MyDrive/wakeword_training_artemis


In [34]:
# === Install microWakeWord (kernel-restart-free) ===
# Workarounds for two upstream bugs:
#  1. kahrendt/microWakeWord setup.py has no find_packages() — non-editable
#     install skips the audio/ subpackage. Editable install needs kernel
#     restart, breaks Run All. Fix: install deps + sys.path.insert().
#  2. train.py calls .numpy() on values that newer TF returns as numpy
#     arrays already. Patch with hasattr() guard.
import os, sys, subprocess, importlib, re

DEPS = [
    'audiomentations', 'audio_metadata', 'datasets', 'mmap_ninja', 'numpy',
    'pymicro-features', 'pyyaml', 'tensorflow>=2.16', 'webrtcvad-wheels',
    'ai-edge-litert',
    'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f',
]
print('Installing dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + DEPS, check=True)

if not os.path.exists('microWakeWord'):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/kahrendt/microWakeWord'], check=True)

MWW_DIR = '/content/microWakeWord'
if MWW_DIR not in sys.path:
    sys.path.insert(0, MWW_DIR)
importlib.invalidate_caches()

fp = '/content/microWakeWord/microwakeword/train.py'
src = open(fp).read()
patched = re.sub(
    r'(\b[a-zA-Z_]+\["[a-z]+"\])\.numpy\(\)',
    r'(\1.numpy() if hasattr(\1, "numpy") else \1)',
    src
)
n = patched.count('hasattr') - src.count('hasattr')
if n > 0:
    open(fp, 'w').write(patched)
    print(f'Patched {n} .numpy() calls in train.py')

import microwakeword
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
print('OK: microwakeword.audio.* imports clean')


Installing dependencies...
OK: microwakeword.audio.* imports clean


In [8]:
# === Data preparation ===
# Mode-aware: either unzip user's bundle, or generate samples inline via Piper.
import os, zipfile

os.chdir('/content')

if MODE == 'bundle':
    print(f'Extracting {BUNDLE_PATH}...')
    with zipfile.ZipFile(BUNDLE_PATH, 'r') as zf:
        zf.extractall('/content')
    for d in ['generated_samples', 'real_recordings', 'confusable_negatives']:
        p = f'/content/{d}'
        if os.path.exists(p):
            n = sum(1 for _ in os.scandir(p) if _.name.endswith('.wav'))
            print(f'  {d}: {n} WAVs')
        else:
            print(f'  {d}: MISSING (will train without)')

elif MODE == 'generate':
    # Defer to the Piper sample-gen cells below
    print('MODE=generate — Piper sample-gen cells will produce samples')

else:
    raise ValueError(f'Unknown MODE: {MODE!r}. Use "bundle" or "generate".')


MODE=generate — Piper sample-gen cells will produce samples


In [36]:
# === Piper sample generator install (skipped if MODE=bundle) ===
if MODE == 'generate':
    import glob, os, shutil, subprocess, sys, urllib.request
    PIPER_REPO_DIR = '/content/piper'
    PIPER_SAMPLE_GENERATOR_DIR = '/content/piper-sample-generator'

    subprocess.run(['apt-get', '-qq', 'install', '-y', 'espeak-ng'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                    'pip', 'setuptools', 'wheel', 'cython'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                    'piper-tts', 'piper-sample-generator'], check=True)

    if not os.path.exists(PIPER_REPO_DIR):
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/rhasspy/piper', PIPER_REPO_DIR], check=True)
    if not os.path.exists(PIPER_SAMPLE_GENERATOR_DIR):
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/rhasspy/piper-sample-generator',
                        PIPER_SAMPLE_GENERATOR_DIR], check=True)

    PIPER_PYTHON_DIR = f'{PIPER_REPO_DIR}/src/python'
    MA_DIR = f'{PIPER_PYTHON_DIR}/piper_train/vits/monotonic_align'
    MA_IMPORT_DIR = f'{MA_DIR}/monotonic_align'
    MA_BUILD_DIR = f'{MA_DIR}/piper_train/vits/monotonic_align'

    shutil.rmtree(f'{PIPER_PYTHON_DIR}/build', ignore_errors=True)
    shutil.rmtree(MA_IMPORT_DIR, ignore_errors=True)
    shutil.rmtree(f'{MA_DIR}/piper_train', ignore_errors=True)
    os.makedirs(MA_IMPORT_DIR, exist_ok=True)
    os.makedirs(MA_BUILD_DIR, exist_ok=True)
    open(f'{MA_IMPORT_DIR}/__init__.py', 'a').close()
    subprocess.run(f'cd {MA_DIR} && {sys.executable} setup.py build_ext --inplace',
                   shell=True, check=True)
    built = next(iter(glob.glob(f'{MA_BUILD_DIR}/core.*')), None)
    assert built, 'monotonic_align core extension build failed'
    shutil.copy2(built, MA_IMPORT_DIR)

    for path in (PIPER_PYTHON_DIR, PIPER_SAMPLE_GENERATOR_DIR):
        if path not in sys.path:
            sys.path.insert(0, path)

    MODEL_PATH = 'models/en_US-libritts_r-medium.pt'
    MODEL_CONFIG_PATH = f'{MODEL_PATH}.json'
    os.makedirs('models', exist_ok=True)
    if not os.path.exists(MODEL_PATH):
        print('Downloading libritts_r model (~75 MB)...')
        urllib.request.urlretrieve(
            'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt',
            MODEL_PATH)
        urllib.request.urlretrieve(
            'https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/libritts_r/medium/en_US-libritts_r-medium.onnx.json',
            MODEL_CONFIG_PATH)
    print('Piper ready')
else:
    print('Skipped (MODE != generate)')


Piper ready


In [37]:
# === Generate positive samples (US English, optionally UK) ===
if MODE == 'generate':
    import os, subprocess, sys
    PIPER_SAMPLE_GENERATOR_DIR = '/content/piper-sample-generator'
    MODEL_PATH = 'models/en_US-libritts_r-medium.pt'
    PIPER_BATCH = 256  # drop to 128 on T4 if CUDA OOM

    def run_piper(target_word, max_samples, output_dir):
#        cmd = [sys.executable, f'{PIPER_SAMPLE_GENERATOR_DIR}/generate_samples.py',
        cmd = [sys.executable, '-m', 'piper_sample_generator',
               target_word, '--phoneme-input', '--model', MODEL_PATH,
               '--max-samples', str(max_samples),
               '--batch-size', str(PIPER_BATCH),
               '--noise-scales', '0.5', '--noise-scale-ws', '0.6',
               '--output-dir', output_dir]
#        subprocess.run(cmd, check=True)
        env = os.environ.copy()
        env["PYTHONPATH"] = (
          "/content/piper-sample-generator"
          + os.pathsep
          + env.get("PYTHONPATH", "")
        )

        subprocess.run(cmd, check=True, env=env)

#        result = subprocess.run(cmd, capture_output=True, text=True)

#        print("STDOUT:")
#        print(result.stdout)
#        print("STDERR:")
#        print(result.stderr)
#        print("Exit code:", result.returncode)

#        result.check_returncode()





    os.makedirs('generated_samples', exist_ok=True)

    print(f'Generating {SAMPLES_US} US samples for {WAKE_WORD_IPA_US!r}...')
    run_piper(WAKE_WORD_IPA_US, SAMPLES_US, 'generated_samples')

    if WAKE_WORD_IPA_UK and SAMPLES_UK > 0:
        print(f'Generating {SAMPLES_UK} UK samples for {WAKE_WORD_IPA_UK!r}...')
        run_piper(WAKE_WORD_IPA_UK, SAMPLES_UK, 'generated_samples')

    n = sum(1 for f in os.listdir('generated_samples') if f.endswith('.wav'))
    print(f'Total positive samples: {n}')
else:
    print('Skipped (MODE != generate)')


Generating 30000 US samples for 'ˈɑɹtəmɪs'...
Generating 6000 UK samples for 'ˈɑɹtɪmɪs'...
Total positive samples: 30000


In [38]:
# === Generate confusable negatives ===
if MODE == 'generate':
    import os, subprocess, sys
    from pathlib import Path
    PIPER_SAMPLE_GENERATOR_DIR = '/content/piper-sample-generator'
    MODEL_PATH = 'models/en_US-libritts_r-medium.pt'

    os.makedirs('confusable_negatives', exist_ok=True)
    for phrase in CONFUSABLE_PHRASES:
        safe = phrase.replace(' ', '_').replace(',', '').replace("'", '')
        existing = len(list(Path('confusable_negatives').glob(f'{safe}_*.wav')))
        if existing >= SAMPLES_PER_CONFUSABLE:
            print(f'  {phrase!r}: {existing} already, skip')
            continue
        tmp = f'/tmp/confusable_{safe}'
        os.makedirs(tmp, exist_ok=True)
        print(f'  generating {SAMPLES_PER_CONFUSABLE} for {phrase!r}...')

        env = os.environ.copy()
        env['PYTHONPATH'] = PIPER_SAMPLE_GENERATOR_DIR

        subprocess.run([
          sys.executable,
          '-m', 'piper_sample_generator',
          phrase,
          '--model', MODEL_PATH,
          '--max-samples', str(SAMPLES_PER_CONFUSABLE),
          '--batch-size', '256',
          '--noise-scales', '0.5',
          '--noise-scale-ws', '0.6',
          '--output-dir', tmp
        ], check=True, env=env)

        # Move + rename with safe prefix
        for f in os.listdir(tmp):
            if f.endswith('.wav'):
                os.rename(f'{tmp}/{f}', f'confusable_negatives/{safe}_{f}')

    n = sum(1 for f in os.listdir('confusable_negatives') if f.endswith('.wav'))
    print(f'Total confusable negatives: {n}')
else:
    print('Skipped (MODE != generate)')


  generating 1000 for 'hey there'...
  generating 1000 for 'hey you'...
  generating 1000 for "hey y'all"...
  generating 1000 for 'hey now'...
  generating 1000 for 'Arthur'...
  generating 1000 for 'Artie'...
  generating 1000 for 'Artist'...
  generating 1000 for 'Artists'...
  generating 1000 for 'Artistic'...
  generating 1000 for 'Art is'...
  generating 1000 for 'Artem'...
  generating 1000 for 'Our team'...
  generating 1000 for 'Our teams'...
  generating 1000 for 'Our time'...
  generating 1000 for 'Are they missing'...
  generating 1000 for 'Part of this'...
  generating 1000 for 'Pardon me'...
  generating 1000 for 'Hard to miss'...
  generating 1000 for 'Hard times'...
Total confusable negatives: 19000


In [26]:
# === Replacement FMA Background Music Download ===
# Downloads FMA Extra Small and converts it to 16 kHz mono WAV files.

import subprocess
from pathlib import Path

base = Path("/content")
source = base / "fma_source"
destination = base / "fma_16k"
archive = source / "fma_xs.zip"

source.mkdir(exist_ok=True)
destination.mkdir(exist_ok=True)

url = "https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/fma_xs.zip"

# Download
if not archive.exists():
    print("Downloading FMA Extra Small...")
    subprocess.run(
        ["wget", "-O", str(archive), url],
        check=True
    )

# Extract
print("Extracting music...")
subprocess.run(
    ["unzip", "-oq", str(archive), "-d", str(source)],
    check=True
)

# Find source audio
audio_files = [
    p for p in source.rglob("*")
    if p.suffix.lower() in (".mp3", ".wav", ".flac", ".ogg")
]

print(f"Found {len(audio_files)} audio files")

# Convert to 16 kHz mono WAV
for i, audio in enumerate(audio_files, 1):
    relative = audio.relative_to(source)
    output = destination / (
        "_".join(relative.with_suffix("").parts) + ".wav"
    )

    if output.exists():
        continue

    subprocess.run(
        [
            "ffmpeg", "-nostdin", "-loglevel", "error",
            "-y", "-i", str(audio),
            "-ar", "16000", "-ac", "1",
            str(output)
        ],
        check=True
    )

    if i % 100 == 0:
        print(f"Converted {i}/{len(audio_files)}")

print("Complete!")
print("WAV files:", len(list(destination.glob("*.wav"))))

Extracting music...
Found 210 audio files
Converted 100/210
Converted 200/210
Complete!
WAV files: 210


In [27]:
# === Download MIT Room Impulse Responses ===
# Preconverted 16 kHz WAV files for room/reverb augmentation

import sys
import subprocess
from pathlib import Path

try:
    from huggingface_hub import snapshot_download
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"],
        check=True
    )
    from huggingface_hub import snapshot_download

destination = Path("/content/mit_rirs")
destination.mkdir(parents=True, exist_ok=True)

print("Downloading MIT room impulse responses...")

snapshot_download(
    repo_id="davidscripka/MIT_environmental_impulse_responses",
    repo_type="dataset",
    allow_patterns=["*.wav", "**/*.wav"],
    local_dir=str(destination),
)

wav_files = list(destination.rglob("*.wav"))

print("Complete!")
print(f"RIR WAV files: {len(wav_files)}")

assert wav_files, "No RIR WAV files found!"

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 270 files:   0%|          | 0/270 [00:00<?, ?it/s]

Complete!
RIR WAV files: 270


In [28]:
# === Environmental Sound Augmentation (FSD50K) ===

import sys
import subprocess
from pathlib import Path

# Install dataset reader if needed
try:
    from datasets import load_dataset
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "datasets"],
        check=True
    )
    from datasets import load_dataset

destination = Path("/content/audioset_16k")
destination.mkdir(parents=True, exist_ok=True)

target_count = 300

print("Streaming FSD50K environmental sounds...")

dataset = load_dataset(
    "quinnlue/FSD50K-16k",
    split="train",
    streaming=True
)

# Preserve encoded audio without requiring automatic decoding
dataset = dataset.decode(False)

count = len(list(destination.glob("*.flac")))

for sample in dataset:
    if count >= target_count:
        break

    audio = sample["audio"]
    audio_bytes = audio.get("bytes")

    if not audio_bytes:
        continue

    output = destination / f"fsd_{count:04d}.flac"
    output.write_bytes(audio_bytes)
    count += 1

print("Complete!")
print(f"Environmental audio files: {count}")

Streaming FSD50K environmental sounds...


README.md:   0%|          | 0.00/4.18k [00:00<?, ?B/s]

Complete!
Environmental audio files: 300


In [29]:
# === Convert FSD50K background sounds to WAV ===

from pathlib import Path
import subprocess

folder = Path("/content/audioset_16k")
flac_files = list(folder.glob("*.flac"))

print(f"Found {len(flac_files)} FLAC files")

for i, src in enumerate(flac_files, 1):
    dst = src.with_suffix(".wav")

    if not dst.exists():
        subprocess.run([
            "ffmpeg", "-nostdin", "-loglevel", "error",
            "-y", "-i", str(src),
            "-ar", "16000", "-ac", "1",
            str(dst)
        ], check=True)

    if i % 100 == 0:
        print(f"Converted {i}/{len(flac_files)}")

wav_files = list(folder.glob("*.wav"))
print(f"Ready: {len(wav_files)} WAV files")

assert len(wav_files) >= 300, "Expected 300 environmental WAV files"

Found 300 FLAC files
Converted 100/300
Converted 200/300
Converted 300/300
Ready: 300 WAV files


In [30]:
# === Download Standard microWakeWord Negative Features ===
# Replaces the original broken dataset download cell.
# Does NOT reinstall or modify microWakeWord.

from pathlib import Path
from huggingface_hub import hf_hub_download
import zipfile
import shutil

destination = Path("/content/negative_datasets")
destination.mkdir(parents=True, exist_ok=True)

archives = [
    "speech.zip",
    "dinner_party.zip",
    "no_speech.zip",
    "dinner_party_eval.zip",
]

repo = "kahrendt/microwakeword"

# Check available space before downloading
usage = shutil.disk_usage("/content")
print(f"Available disk space: {usage.free / (1024**3):.1f} GB")

for filename in archives:
    name = filename.removesuffix(".zip")
    expected = destination / name

    if expected.is_dir() and any(expected.rglob("*")):
        print(f"{name}: already extracted, skipping")
        continue

    print(f"\nDownloading {filename}...")

    archive_path = hf_hub_download(
        repo_id=repo,
        repo_type="dataset",
        filename=filename,
    )

    print(f"Extracting {filename}...")

    with zipfile.ZipFile(archive_path) as zf:
        # Inspect archive structure
        print("First archive entries:", zf.namelist()[:3])
        zf.extractall(destination)

    print(f"{name}: extraction finished")

print("\n=== Dataset verification ===")

for filename in archives:
    name = filename.removesuffix(".zip")
    folder = destination / name

    if folder.is_dir():
        files = [p for p in folder.rglob("*") if p.is_file()]
        print(f"{name}: {len(files)} files")
    else:
        print(f"{name}: expected folder NOT FOUND")

print("\nDownload/extraction stage complete.")

Available disk space: 190.9 GB



speech.zip: reconstructing file:   0%|          |  0.00B / 3.18GB            

speech.zip: downloading bytes:           |  0.00B            

Extracting speech.zip...
First archive entries: ['speech/', 'speech/training/', 'speech/training/voices_lav_mid_training_mmap/']
speech: extraction finished



dinner_party.zip: reconstructing file:   0%|          |  0.00B /  444MB            

...

no_speech.zip: downloading bytes:           |  0.00B            

Extracting no_speech.zip...
First archive entries: ['no_speech/', 'no_speech/training/', 'no_speech/training/fma_medium_mmap/']
no_speech: extraction finished



dinner_party_eval.zip: reconstructing file:   0%|          |  0.00B / 82.3MB            

dinner_party_eval.zip: downloading bytes:           |  0.00B            

Extracting dinner_party_eval.zip...
First archive entries: ['dinner_party_eval/', 'dinner_party_eval/testing_ambient/', 'dinner_party_eval/testing_ambient/dipco_u01_ch1_mmap/']
dinner_party_eval: extraction finished

=== Dataset verification ===
speech: 141 files
dinner_party: 94 files
no_speech: 188 files
dinner_party_eval: 94 files

Download/extraction stage complete.


In [40]:
# === Augmentation + feature extraction ===
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
import os, shutil, traceback
from mmap_ninja.ragged import RaggedMmap

clips = Clips(
    input_directory='generated_samples',
    file_pattern='*.wav',
    max_clip_duration_s=None,
    remove_silence=True,
    random_split_seed=42,
    split_count=0.1,
)
augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        'SevenBandParametricEQ': 0.15, 'TanhDistortion': 0.10,
        'PitchShift': 0.15, 'BandStopFilter': 0.10,
        'AddColorNoise': 0.20, 'AddBackgroundNoise': 0.85,
        'Gain': 1.00, 'GainTransition': 0.25, 'RIR': 0.60,
    },
    impulse_paths=['mit_rirs'],
    background_paths=['fma_16k', 'audioset_16k'],
    background_min_snr_db=-5, background_max_snr_db=20,
    min_jitter_s=0.10, max_jitter_s=0.50,
)

os.makedirs('generated_augmented_features', exist_ok=True)
SPLIT_CONFIG = {
    'training':   {'split_name': 'train',      'repetition': 3, 'slide_frames': 10},
    'validation': {'split_name': 'validation', 'repetition': 1, 'slide_frames': 10},
    'testing':    {'split_name': 'test',       'repetition': 1, 'slide_frames': 1 },
}

for split, cfg in SPLIT_CONFIG.items():
    out = f'generated_augmented_features/{split}'
    mmap = f'{out}/wakeword_mmap'
    if os.path.exists(mmap) and list(os.scandir(mmap)):
        print(f'{split}: cached, skipping')
        continue
    if os.path.exists(mmap):
        shutil.rmtree(mmap)
    os.makedirs(out, exist_ok=True)
    print(f'Generating {split} (rep={cfg["repetition"]}, slide={cfg["slide_frames"]})...')
    try:
        sg = SpectrogramGeneration(clips=clips, augmenter=augmenter,
                                    slide_frames=cfg['slide_frames'], step_ms=10)
        RaggedMmap.from_generator(
            out_dir=mmap, batch_size=200, verbose=True,
            sample_generator=sg.spectrogram_generator(
                split=cfg['split_name'], repeat=cfg['repetition']),
        )
    except Exception:
        traceback.print_exc()
        if os.path.exists(mmap): shutil.rmtree(mmap)
        raise
print('Positive features ready')

# Confusable features (only if confusable_negatives/ exists)
if os.path.exists('confusable_negatives') and os.listdir('confusable_negatives'):
    print('Generating confusable features...')
    confusable_clips = Clips(
        input_directory='confusable_negatives', file_pattern='*.wav',
        max_clip_duration_s=None, remove_silence=True,
        random_split_seed=42, split_count=0.1,
    )
    os.makedirs('confusable_features', exist_ok=True)
    for split, cfg in SPLIT_CONFIG.items():
        out = f'confusable_features/{split}'
        mmap = f'{out}/wakeword_mmap'
        if os.path.exists(mmap) and list(os.scandir(mmap)):
            continue
        if os.path.exists(mmap): shutil.rmtree(mmap)
        os.makedirs(out, exist_ok=True)
        sg = SpectrogramGeneration(clips=confusable_clips, augmenter=augmenter,
                                    slide_frames=cfg['slide_frames'], step_ms=10)
        RaggedMmap.from_generator(
            out_dir=mmap, batch_size=200, verbose=True,
            sample_generator=sg.spectrogram_generator(
                split=cfg['split_name'], repeat=cfg['repetition']),
        )
    print('Confusable features ready')

# Real recording features (only if real_recordings/ exists)
if os.path.exists('real_recordings') and os.listdir('real_recordings'):
    print('Generating real-recording features...')
    real_clips = Clips(
        input_directory='real_recordings', file_pattern='*.wav',
        max_clip_duration_s=None, remove_silence=True,
        random_split_seed=42, split_count=0.1,
    )
    os.makedirs('real_recording_features', exist_ok=True)
    for split, cfg in SPLIT_CONFIG.items():
        out = f'real_recording_features/{split}'
        mmap = f'{out}/wakeword_mmap'
        if os.path.exists(mmap) and list(os.scandir(mmap)):
            continue
        if os.path.exists(mmap): shutil.rmtree(mmap)
        os.makedirs(out, exist_ok=True)
        sg = SpectrogramGeneration(clips=real_clips, augmenter=augmenter,
                                    slide_frames=cfg['slide_frames'], step_ms=10)
        RaggedMmap.from_generator(
            out_dir=mmap, batch_size=200, verbose=True,
            sample_generator=sg.spectrogram_generator(
                split=cfg['split_name'], repeat=cfg['repetition']),
        )
    print('Real-recording features ready')


Generating training (rep=3, slide=10)...


0it [00:00, ?it/s]

Generating validation (rep=1, slide=10)...


0it [00:00, ?it/s]

Generating testing (rep=1, slide=1)...


...

Positive features ready
Generating confusable features...


0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

Confusable features ready


In [41]:
# === Training config YAML ===
import yaml, os
from pathlib import Path

SKIP_CONFUSABLES = not Path('confusable_features/training/wakeword_mmap').exists()
SKIP_REAL = not Path('real_recording_features/training/wakeword_mmap').exists()

config = {
    'window_step_ms': 10,
    'train_dir': f'trained_models/{OUTPUT_NAME}',
    'features': [
        dict(features_dir='generated_augmented_features', sampling_weight=8.0,
             penalty_weight=2.0, truth=True, truncation_strategy='truncate_start',
             type='mmap'),
        dict(features_dir='negative_datasets/speech', sampling_weight=10.0,
             penalty_weight=2.5, truth=False, truncation_strategy='random', type='mmap'),
        dict(features_dir='negative_datasets/dinner_party', sampling_weight=15.0,
             penalty_weight=3.0, truth=False, truncation_strategy='random', type='mmap'),
        dict(features_dir='negative_datasets/no_speech', sampling_weight=5.0,
             penalty_weight=1.0, truth=False, truncation_strategy='random', type='mmap'),
        dict(features_dir='negative_datasets/dinner_party_eval', sampling_weight=0.0,
             penalty_weight=1.0, truth=False, truncation_strategy='split', type='mmap'),
    ],
    'training_steps': [25000, 20000],
    #'training_steps': [100, 100],
    'positive_class_weight': [2, 2],
    'negative_class_weight': [40, 50],
    'learning_rates': [0.001, 0.0001],
    'batch_size': 256,
    'time_mask_max_size': [5, 5], 'time_mask_count': [1, 1],
    'freq_mask_max_size': [3, 3], 'freq_mask_count': [1, 1],
    'eval_step_interval': 500,
    #'eval_step_interval': 50,
    'clip_duration_ms': 1500,
    'target_minimization': 0.4,
    'minimization_metric': 'ambient_false_positives_per_hour',
    'maximization_metric': 'average_viable_recall',
}

if not SKIP_CONFUSABLES:
    config['features'].append(dict(features_dir='confusable_features', sampling_weight=8.0,
                                    penalty_weight=5.0, truth=False,
                                    truncation_strategy='random', type='mmap'))
if not SKIP_REAL:
    config['features'].append(dict(features_dir='real_recording_features', sampling_weight=8.0,
                                    penalty_weight=2.0, truth=True,
                                    truncation_strategy='truncate_start', type='mmap'))

os.makedirs(f'trained_models/{OUTPUT_NAME}', exist_ok=True)
with open('training_parameters.yaml', 'w') as f:
    yaml.dump(config, f)
print('training_parameters.yaml ready')
print(f'  Feature sets: {len(config["features"])}, total steps: {sum(config["training_steps"])}')


training_parameters.yaml ready
  Feature sets: 6, total steps: 45000


In [42]:
# === Train the model ===
# subprocess + PYTHONPATH so it can find microwakeword (sys.path doesn't propagate)
import os, sys, subprocess, shutil
shutil.rmtree(f'trained_models/{OUTPUT_NAME}', ignore_errors=True)

env = os.environ.copy()
env['PYTHONPATH'] = '/content/microWakeWord:' + env.get('PYTHONPATH', '')
env['XLA_FLAGS'] = '--xla_gpu_autotune_level=0'

cmd = [
    sys.executable, '-m', 'microwakeword.model_train_eval',
    '--training_config', 'training_parameters.yaml',
    '--train', '1', '--restore_checkpoint', '0',
    '--test_tflite_streaming_quantized', '1',
    '--use_weights', 'best_weights',
    'mixednet',
    '--pointwise_filters', '64,64,64,64',
    '--repeat_in_block', '1, 1, 1, 1',
    '--mixconv_kernel_sizes', '[5], [7,11], [9,15], [23]',
    '--residual_connection', '0,0,0,0',
    '--first_conv_filters', '32',
    '--first_conv_kernel_size', '5',
    '--stride', '3',
]
print('Running:', ' '.join(cmd)); print()
proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print(); print('Exit code:', proc.returncode)
assert proc.returncode == 0, 'training failed - see output above'


Streaming output truncated to the last 5000 lines.
Validation Batch #81: Accuracy = 0.999; Recall = 0.998; Precision = 0.999; Loss = 0.0019; Mini-Batch #360
Validation Batch #81: Accuracy = 0.999; Recall = 0.998; Precision = 0.999; Loss = 0.0019; Mini-Batch #361
Validation Batch #81: Accuracy = 0.999; Recall = 0.998; Precision = 0.999; Loss = 0.0019; Mini-Batch #362
Validation Batch #81: Accuracy = 0.999; Recall = 0.998; Precision = 0.999; Loss = 0.0019; Mini-Batch #363
Validation Batch #81: Accuracy = 0.999; Recall = 0.998; Precision = 0.999; Loss = 0.0019; Mini-Batch #364
Validation Batch #81: Accuracy = 0.999; Recall = 0.998; Precision = 0.999; Loss = 0.0019; Mini-Batch #365
Validation Batch #81: Accuracy = 0.999; Recall = 0.998; Precision = 0.999; Loss = 0.0019; Mini-Batch #366
Validation Batch #81: Accuracy = 0.999; Recall = 0.998; Precision = 0.999; Loss = 0.0019; Mini-Batch #367
Validation Batch #81: Accuracy = 0.999; Recall = 0.998; Precision = 0.999; Loss = 0.0019; Mini-Batch 

In [43]:
# === Export + push to Drive ===
import os, json, shutil, datetime

tflite_src = f'trained_models/{OUTPUT_NAME}/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite'
assert os.path.exists(tflite_src), f'No model at {tflite_src}'

OUT_TFLITE = f'{OUTPUT_NAME}.tflite'
OUT_JSON = f'{OUTPUT_NAME}.json'
shutil.copy2(tflite_src, OUT_TFLITE)
print(f'wrote {OUT_TFLITE} ({os.path.getsize(OUT_TFLITE)/1024:.1f} KB)')

manifest = {
    'type': 'micro',
    'wake_word': WAKE_WORD,
    'author': AUTHOR,
    'website': AUTHOR_WEBSITE,
    'model': OUT_TFLITE,
    'trained_languages': TRAINED_LANGUAGES,
    'version': 2,
    'micro': {
        'probability_cutoff': PROBABILITY_CUTOFF,
        'feature_step_size': 10,
        'sliding_window_size': SLIDING_WINDOW_SIZE,
        'tensor_arena_size': TENSOR_ARENA_SIZE,
        'minimum_esphome_version': '2024.7.0',
    }
}
with open(OUT_JSON, 'w') as f:
    json.dump(manifest, f, indent=2)
print(f'wrote {OUT_JSON}')
print(json.dumps(manifest, indent=2))

for fn in (OUT_TFLITE, OUT_JSON):
    shutil.copy2(fn, f'{DRIVE_DIR}/{fn}')
    print(f'pushed {fn} -> {DRIVE_DIR}')

ts = datetime.datetime.now(datetime.timezone.utc).isoformat()
with open(f'{DRIVE_DIR}/_run_finished.txt', 'w') as f:
    f.write(f'Training run finished at {ts}\n')
print()
print(f'DONE. Find your model at /content/drive/MyDrive/{DRIVE_FOLDER}/{OUTPUT_NAME}.tflite')


wrote artemis.tflite (60.8 KB)
wrote artemis.json
{
  "type": "micro",
  "wake_word": "Artemis",
  "author": "lizaoreo",
  "website": "https://github.com/lizaoreo",
  "model": "artemis.tflite",
  "trained_languages": [
    "en"
  ],
  "version": 2,
  "micro": {
    "probability_cutoff": 0.85,
    "feature_step_size": 10,
    "sliding_window_size": 5,
    "tensor_arena_size": 50000,
    "minimum_esphome_version": "2024.7.0"
  }
}
pushed artemis.tflite -> /content/drive/MyDrive/wakeword_training_artemis
pushed artemis.json -> /content/drive/MyDrive/wakeword_training_artemis

DONE. Find your model at /content/drive/MyDrive/wakeword_training_artemis/artemis.tflite


## Deploying to ESPHome devices

Drop the `.tflite` + `.json` next to your ESPHome YAML (e.g. in `/config/esphome/wakewords/<output_name>/`).

In your device YAML, replace your existing wake word:

```yaml
micro_wake_word:
  models:
    - model: wakewords/<output_name>/<output_name>.json
  on_wake_word_detected:
    - voice_assistant.start:
```

Then `esphome run <device>.yaml` (USB or OTA).

## Manifest tuning (likely needed)

The defaults work for **Hey Harold** specifically. Your model's confidence
distribution will differ. Iterate:

| Symptom | Knob |
|---|---|
| Doesn't fire on the wake word | Lower `probability_cutoff` (try 0.7, 0.6) |
| Fires on too many things | Raise `probability_cutoff` (try 0.92, 0.95) |
| LED fires but no STT response | `sliding_window_size: 5` (faster fire); also check Echo speaker mute |
| `Failed to allocate tensors` log | Raise `tensor_arena_size` (try 50000, 80000) |

## Known gotchas
- Editable install (`pip install -e ./microWakeWord`) requires kernel restart — broken for Run All
- `train.py` upstream calls `.numpy()` on numpy arrays under newer TF (this notebook patches it)
- T4 GPU OOMs during validation — use A100 + High-RAM
- Manifest path mismatch: training writes to `trained_models/<output_name>/`, manifest must match
- `voice_assistant` has no audio lookback; if MWW fires AFTER user speaks, STT gets silence
